# 04 · Baselines, LightGBM ablation and holdout evaluation (CP3)

The question this notebook answers:

> does weather, hotspot detections and upwind exposure add anything to **CAMS history + weather +
> time** when predicting the **CAMS model field at t+6 h** on one grid cell?

What the numbers are, and are not:

- The target is the **CAMS model field**, not observed PM2.5. The snapshot has no `issued_at`, so no
  issue-time baseline can be rebuilt, and nothing here is an operational forecast score.
- The run is written as `research_only: true`, `operational_ready: false`,
  `observation_validated: false`. A holdout score against a model field is not station validation.
- The split is chronological with a 6 h gap on both boundaries: train 2024, validation 2025, test
  from 2026-01-01. Test rows are never used to fit, to stop or to choose anything.
- Baseline B here is a **train-only month/hour mean**, not the product methodology's CAMS issue-time
  baseline. That one needs an issue time the snapshot does not have, so it is not run.

| ID | Model | Features |
| --- | --- | --- |
| A | persistence | CAMS model field at `t` |
| B | train-only month/hour mean of the target | none (group means) |
| C | LightGBM | CAMS history + weather + wind + time |
| D | LightGBM | C + detection counts and FRP sums |
| E | LightGBM | D + upwind exposure |

In [ ]:
from pavana_ml.models.experiment import (
    CALENDAR_BREAKDOWN_RULE,
    DEFAULT_TEST_START,
    DEFAULT_VALIDATION_START,
    artifact_status,
    chronological_split,
    load_model_table,
    run_experiment,
)
from pavana_ml.models.lgbm import latest_app_ready_artifact, latest_evaluated_artifact
from pavana_ml.paths import ARTIFACTS_DIR

table, metadata = load_model_table()
target = metadata["target"]
eligibility = metadata["eligibility"]

print(f"table          : {metadata['table_path']}")
print(f"  rows         : {len(table)}  eligible: {eligibility['eligible']}")
print(f"  sha256       : {metadata['table_sha256']}")
print(f"target         : {target}  (kind: {metadata['target_kind']})")
print(
    "table status   : "
    f"research_only={metadata['research_only']}, "
    f"operational_ready={metadata['operational_ready']}, "
    f"observation_validated={metadata['observation_validated']}"
)
print()
print("The stored table matched its recorded SHA256, so every number below describes this table.")
print("A mismatch stops the run instead of producing a metric for an unknown file.")

## Split

| Split | Origin period | Must end before | Use |
| --- | --- | --- | --- |
| train | 2024 | `2025-01-01 UTC − 6 h` | fit the trees and baseline B's group means |
| validation | 2025 | `2026-01-01 UTC − 6 h` | early stopping only |
| test | from `2026-01-01 UTC` | last available target | final evaluation |

The cut is on the **origin** hour, and `max(target_valid_at)` of a split must stay below the first
origin of the next one. That is why the 6 h band before each boundary belongs to no split, and why
`chronological_split` refuses an empty window instead of reporting `NaN` as a score.

In [ ]:
splits, windows = chronological_split(table, metadata)

for name, window in windows["splits"].items():
    origins = f"{window['first_origin_utc']} .. {window['last_origin_utc']}"
    labels = f"{window['first_target_valid_utc']} .. {window['last_target_valid_utc']}"
    print(f"{name:>10}: {window['rows']:>5} rows   origins {origins}")
    print(f"{' ':>10}  labels  {labels}")
print()
print(f"eligible rows dropped inside the {windows['gap_hours']} h gap: {windows['dropped_in_gap']}")
print(
    "ineligible rows kept in the table (48 h warm-up, 6 h tail, missing history): "
    f"{windows['ineligible_rows']}"
)
print()
for name, stats in windows["target_distribution_ug_m3"].items():
    print(
        f"target {name:>10}: min {stats['min']:6.1f}  p50 {stats['p50']:6.1f}  "
        f"max {stats['max']:6.1f}  µg/m³"
    )

### Reading the level shift

The block above reports the model field's own level in each split, because it decides how the
metrics below should be read. A boosted tree can only output values inside the range it was trained
on, so when the test target climbs well above the train range the tree variants flatten out there,
while persistence keeps carrying the current level. That is a property of the split and of the
target, not of the hotspot features, and it is reported here rather than tuned away.

## What is fixed before the test split is read

- The variant named in the report is the one with the lowest **validation** MAE, ties broken by
  fewer features. The test metrics of every variant are written either way, so a losing variant
  cannot be hidden by that choice, and no test number may change a model, a feature list or a
  parameter afterwards.
- LightGBM keeps `NaN` as its own branch, so no imputation or scaling is fitted at all. The only
  fitted statistics are the boosted trees and baseline B's train-only month/hour means.
- Early stopping watches the validation split, on CPU, with an explicit seed, thread count and
  deterministic flags. The run records every one of them next to the metrics.
- The feature lists are the C/D/E allowlists from the table metadata, not "every numeric column".
  The target, `target_valid_at` and `eligible` are rejected if they appear in one.

In [ ]:
RUN_VERSION = "cams-t6h-retro-v1"

run = run_experiment(
    table,
    metadata,
    validation_start=DEFAULT_VALIDATION_START,
    test_start=DEFAULT_TEST_START,
    max_rounds=500,
    version=RUN_VERSION,
    if_exists="reuse",
)
results, test_predictions, report = run["results"], run["test_predictions"], run["report"]
test_rows = results[(results["split"] == "test") & (results["period"] == "all")]

print(f"run            : {run['output_dir']}")
print(f"version        : {report['version']}")
print("                 (a finished run is reused only when its table, code and settings match)")
print(f"selected       : {report['selected_variant']} on {report['selected_on']}")
print(f"validation MAE : {report['validation_mae']}")
print(f"best iteration : {report['lightgbm']['best_iteration']}")
print()
print("Holdout metrics. Every variant is scored on the identical eligible test rows:")
print(test_rows[["variant", "features_n", "n", "mae", "rmse"]].to_string(index=False))

In [ ]:
by_split = results[results["period"] == "all"].pivot(index="variant", columns="split", values="mae")
print("MAE by split — train rows are in-sample for every learner,")
print("validation rows stopped the trees:")
print(by_split[["train", "validation", "test"]].round(3).to_string())
print()
lowest = test_rows.sort_values("mae").iloc[0]
print(f"Lowest test MAE: {lowest['variant']} — {lowest['label']} ({lowest['mae']:.2f} µg/m³)")
print()
scored = test_rows.set_index("variant")["mae"]
for base, added in (("C", "D"), ("D", "E")):
    delta = scored[added] - scored[base]
    print(f"{added} − {base}: {delta:+.3f} µg/m³ MAE on the test split")
print()
print("A negative delta means the extra feature group scored lower on this holdout. One holdout")
print("cannot establish skill, and a variant is never chosen on a test number — see the")
print("report's selection rule and validation MAE above.")

In [ ]:
monthly = results[(results["split"] == "test") & (results["period"] != "all")]
print("Test MAE per calendar month (UTC):")
print(monthly.pivot_table(index="period", columns="variant", values="mae").round(2).to_string())
print()
print(CALENDAR_BREAKDOWN_RULE)
print()
events = test_rows[
    ["variant", "event_threshold_ug_m3", "event_support", "event_precision", "event_recall"]
]
print("Hours at or above the event threshold in the test target, and what each variant predicted:")
print(events.to_string(index=False))
print()
print("An undefined precision means the variant never predicted a value at or above the cutoff.")
print(report["event_metric"]["meaning"])

In [ ]:
print(f"artifacts written under {run['output_dir']}:")
for path in sorted(run["output_dir"].rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(run['output_dir'])}")
print()
print(report["status"]["statement"])
print()
print("Model artifacts found under the artifacts root:")
for entry in artifact_status(ARTIFACTS_DIR):
    print(
        f"  {entry['version']}: evaluated={entry['evaluated']} target_kind={entry['target_kind']} "
        f"observation_validated={entry['observation_validated']} app_ready={entry['app_ready']}"
    )
print()
print(f"latest_evaluated_artifact(): {latest_evaluated_artifact()}")
print(f"latest_app_ready_artifact() : {latest_app_ready_artifact()}")
print()
print("A run keeps its models in C/, D/ and E/ subdirectories and writes no top-level")
print("metadata.json, so nothing here is offered as a servable model: `evaluated` is not")
print("`operational_ready`.")

## What this run does not establish

- **Not station validation.** The target is the CAMS model field on one returned grid cell. No
  observed PM2.5 series enters this notebook.
- **Not forecast skill.** Without an `issued_at` there is no lead time: the "prediction" is a
  retrospective fit of a model field.
- **Not an untouched holdout any more, once read.** The test split has been opened here. Any later
  change to features, parameters or the target horizon is a second look at a used holdout and has to
  be reported as such.
- **Not fires.** Detection counts are records in this cache. Zero means no record in the window,
  never proof of satellite coverage, and `upwind_score` is an indicator, not a plume model and not a
  probability that smoke arrives.
- **Not haze episodes.** The month breakdown is a calendar breakdown; no month is labelled an event.
- **Not a tuned result.** A negative outcome (E no better than D or C, or every variant behind
  persistence) is a valid research result and is left as measured.